# TT1 - MDM UBA - 2025

**Tariff classification using NLP**

By Santiago Tedoldi

## Training a DistiltBERT for classification

In [ ]:
# Dependencies
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re
from typing import Sequence, Optional, Dict, Any, Tuple


### Raw dataset

In [2]:
colspecs = [(0, 6), (6, None)]
data_type = {'HS06': str}
df = pd.read_fwf('data/raw_data_HScodes_desc.txt',
                 colspecs=colspecs, header=None,
                 names=['HS06', 'GOODS_DESCRIPTION'],
                 dtype=data_type)

### Quick EDA

null and duplicated samples

dropping duplicates

analyzing tops and bottoms regarding frequencies

In [3]:
# Quick EDA
print("=== Quick EDA ===")

# Add HS02 (chapter) and HS04 (heading)
df['HS04'] = df['HS06'].str[:4]
df['HS02'] = df['HS06'].str[:2]

print("Nulls per column:")
print(df.isnull().sum(), "\n")

print("Duplicate rows:", df.duplicated().sum(), "\n")

# Function to build and display freq tables
def freq_table(col, name):
    vc      = df[col].value_counts().rename('count')
    rel     = df[col].value_counts(normalize=True).rename('rel_freq')
    cum     = rel.cumsum().rename('cum_freq')
    summary = pd.concat([vc, rel, cum], axis=1)
    summary['rel_freq'] = (summary['rel_freq'] * 100).round(2).astype(str) + '%'
    summary['cum_freq'] = (summary['cum_freq'] * 100).round(2).astype(str) + '%'

    print(f"## Samples per {name} ({col})\n")
    print("### Top 10")
    print(summary.head(10).to_markdown(), "\n")
    print("### Bottom 10")
    print(summary.tail(10).to_markdown(), "\n")

# Dropping duplicates
df.drop_duplicates(inplace=True)

# Chapter-level (HS02)
freq_table('HS02', 'chapter')

# Heading-level (HS04)
freq_table('HS04', 'heading')

# Subheading-level (HS06)
freq_table('HS06', 'subheading')

=== Quick EDA ===
Nulls per column:
HS06                 0
GOODS_DESCRIPTION    0
HS04                 0
HS02                 0
dtype: int64 

Duplicate rows: 232220 

## Samples per chapter (HS02)

### Top 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     84 |   54901 | 20.5%      | 20.5%      |
|     85 |   33571 | 12.54%     | 33.04%     |
|     87 |   28476 | 10.63%     | 43.67%     |
|     73 |   16173 | 6.04%      | 49.71%     |
|     39 |   12218 | 4.56%      | 54.28%     |
|     90 |   11611 | 4.34%      | 58.61%     |
|     82 |    7972 | 2.98%      | 61.59%     |
|     94 |    7921 | 2.96%      | 64.55%     |
|     40 |    7526 | 2.81%      | 67.36%     |
|     83 |    4285 | 1.6%       | 68.96%     | 

### Bottom 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     41 |      22 | 0.01%      | 99.96%     |
|     81 |      19 | 0.01%      | 99.97%     |
|     45 |      19 | 0

In [4]:
df

,HS06,GOODS_DESCRIPTION,HS04,HS02
0,271019,BRAKE FLUID DOT 4 50X200ML,2710,27
1,847710,PLASTIC INJECTION MOULD MODEL 21A 110G DSM1010...,8477,84
2,844399,LCD ASSEMBLY,8443,84
3,848280,BEARING 22238 KCAW33C3 BRAND MCB,8482,84
4,630900,USED HANDBAGS AND WALLETS,6309,63
...,...,...,...,...
499959,854239,PCB OPTIONAL ADD. KROPT V4.0 (NEW OUT PUT CARD),8542,85
499961,842091,CYLINDER (SDA80*10F003000001A),8420,84
499970,830249,BEOTIC DEVICE,8302,83
499981,901180,COMPOUND BINOCULAR MICROSCOPE,9011,90


Merging with HS06 nomenclature

In [5]:
df_hs06 = pd.read_csv('data/hs06_full_eng.csv', index_col='hs06', 
                      dtype={'hs06': str, 'full_eng': str},
                      usecols=['hs06', 'full_eng'])

In [6]:
# top 5 rows in HS06 nomemclature
print(df_hs06.head(5).to_markdown(), "\n")

# bottom 5 rows in HS06 nomemclature
print(df_hs06.tail(5).to_markdown(), "\n")

|   hs06 | full_eng                                                                              |
|-------:|:--------------------------------------------------------------------------------------|
| 010120 | Live horses, asses, mules and hinnies. && - Horses :                                  |
| 010121 | Live horses, asses, mules and hinnies. && - Horses : && -- Pure-bred breeding animals |
| 010129 | Live horses, asses, mules and hinnies. && - Horses : && -- Other                      |
| 010130 | Live horses, asses, mules and hinnies. && - Asses                                     |
| 010190 | Live horses, asses, mules and hinnies. && - Other                                     | 

|   hs06 | full_eng                                                                                                                                                                                                                                             |
|-------:|:------------------------------------

In [7]:
df = pd.merge(df, df_hs06,how='left', left_on='HS06', right_on='hs06')

In [8]:
print("Nulls per column:")
print(df.isnull().sum()/len(df), "\n")

Nulls per column:
HS06                 0.000000
GOODS_DESCRIPTION    0.000000
HS04                 0.000000
HS02                 0.000000
full_eng             0.045381
dtype: float64 



There are 4.5 % of goods with no HS full_eng available

They may are not updated codes

### Deep EDA

aggregate text statistics by HS level

performed in HSrecomm_EDA

Utils

In [9]:
# Utils f
def hs_frequencies_process(df, hs_codification = []):
    for hs_codi in hs_codification:

        df = df.merge(df[hs_codi].value_counts(), left_on=hs_codi, right_index=True)
        df.rename(columns={'count':f'{hs_codi}_samples'}, inplace=True)

    return df

def description_length(df, description_cols = []):
    for col in description_cols:

        df[f'{col}_len_words'] = df[col].apply(lambda x: len(x.split()))
        df[f'{col}_len_chars'] = df[col].apply(lambda x: len(x))

    return df

def subtokenization_indicator(description, tokenizer):
    words = description.lower().split()
    tokens = tokenizer.tokenize(description)
    return len(tokens)/len(words)

In [10]:
df.head()

,HS06,GOODS_DESCRIPTION,HS04,HS02,full_eng
0,271019,BRAKE FLUID DOT 4 50X200ML,2710,27,Petroleum oils and oils obtained from bitumino...
1,847710,PLASTIC INJECTION MOULD MODEL 21A 110G DSM1010...,8477,84,Machinery for working rubber or plastics or fo...
2,844399,LCD ASSEMBLY,8443,84,Printing machinery used for printing by means ...
3,848280,BEARING 22238 KCAW33C3 BRAND MCB,8482,84,"Ball or roller bearings. && - Other, including..."
4,630900,USED HANDBAGS AND WALLETS,6309,63,NaN


### Preprocessing of text

In [ ]:
stop_words = {'of', 'or', 'and', 'for', 'than', 'the', 'in', 'with', 'to', 'but', 'by'
             , 'whether', 'on', 'its', 'an', 'their', 'at', 'this', 'which', 'from'
             , 'as', 'be', 'is'}
alphabet_pattern = re.compile(r'[^a-zA-Z]')
alphabet_number_pattern = re.compile(r'[^a-zA-Z0-9]')
remove_pattern = re.compile(r'[\;\,\)\(\[\]\:]')


def refine_text_func(text):
    text = text.lower()
    text = ' '.join([w for w in text.split() if w not in stop_words])
    alphabet = re.sub(alphabet_pattern, ' ', text)
    alphabet_number = re.sub(alphabet_number_pattern, ' ', text)
    remove = re.sub(remove_pattern, ' ', text)
    result = ' '.join([text, alphabet, alphabet_number, remove])
    return result

In [ ]:
df['PREPRO_DESCRIPTION'] = df['GOODS_DESCRIPTION'].progress_apply(lambda x: refine_text_func(x))

### N-gram generation

In [ ]:
def create_ngram_data(text, ngram_value=2):
    text_list = text.split()
    ngram_list = list(zip(*[text_list[i:] for i in range(ngram_value)]))
    result = []
    for n_data in ngram_list:
        result.append('_'.join(n_data))
    return ' '.join(result)

create_ngram_data('LIVE BREEDING FARM HORSE')

In [ ]:
df['NGRAM_DESCRIPTION'] = df['PREPRO_DESCRIPTION'].progress_apply(lambda x: create_ngram_data(x))

In [ ]:
df.head()

### DistilBERT model training

Iteraring to measure stability

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel

Sampling function

In [12]:
def bootstrap_sampling(df, test_fraction=0.1):
    # Determine the number of test samples
    n_test = int(len(df) * test_fraction)
    # Perform bootstrap sampling for the test set
    test_set = df.sample(n=n_test, replace=True)
    # Remove the test samples from the original dataframe to create the training set
    train_set = df.drop(test_set.index)
    
    return train_set, test_set

Dataset & DataLoader preparation

In [ ]:
# Sampling for testing the pipeline
df = df.sample(frac=0.01, random_state=42)

Pre-tokenizacion

In [14]:
class TokenizedDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label': self.labels[idx]
        }

Tokenizer

In [ ]:
# Load the tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

Model class

In [ ]:
class HSClassifier(nn.Module):
    def __init__(self,
                 n_classes: int,
                 fine_tune: bool = False,
                 n_finetune_layers: int = 0):
        """
        Args:
          n_classes:      number of target classes
          fine_tune:      if True, you’ll unfreeze either all or the last layers
          n_finetune_layers:
                          • =0 (default) → if fine_tune=True, unfreeze *all* DistilBERT layers  
                          • >0             → unfreeze only that many of the *last* transformer blocks  
                          • ignored if fine_tune=False (encoder stays fully frozen)
        """
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # Freeze everything by default
        for param in self.distilbert.parameters():
            param.requires_grad = False

        # If fine_tune, decide what to unfreeze
        if fine_tune:
            if n_finetune_layers > 0:
                # Unfreeze only the last `n_finetune_layers` transformer blocks
                for block in self.distilbert.transformer.layer[-n_finetune_layers:]:
                    for param in block.parameters():
                        param.requires_grad = True
            else:
                # n_finetune_layers == 0 → unfreeze *all* DistilBERT params
                for param in self.distilbert.parameters():
                    param.requires_grad = True

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.distilbert.config.hidden_size, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.3),
            nn.Linear(1024, n_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state[:, 0, :]  # Take <CLS> token representation
        logits = self.classifier(hidden_state)
        return logits

Training utils

In [ ]:
from tqdm.auto import tqdm

# Accuracy functions
def accuracy(outputs, labels):
    _, preds = torch.max(outputs, dim=1)
    return torch.sum(preds == labels).item()

def top5_accuracy(outputs, labels):
    top5 = torch.topk(outputs, 5, dim=1).indices
    return sum([labels[i] in top5[i] for i in range(labels.size(0))])

def train_epoch(model, data_loader, criterion, optimizer, device):
    print("Model is training on:", next(model.parameters()).device)
    model.train()

    losses = []
    correct = 0
    correct_top5 = 0

    # wrap your DataLoader in a tqdm iterator
    loop = tqdm(data_loader, desc="Training", leave=False)
    for batch in loop:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs, labels)

        correct      += accuracy(outputs, labels)
        correct_top5 += top5_accuracy(outputs, labels)
        losses.append(loss.item())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # update the tqdm bar with current metrics
        loop.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct/len(data_loader.dataset):.4f}",
            top5=f"{correct_top5/len(data_loader.dataset):.4f}"
        )

    # make sure you end the line so console prompt isn't on the last bar
    print()

    return (
        correct      / len(data_loader.dataset),
        correct_top5 / len(data_loader.dataset),
        sum(losses)  / len(losses)
    )

def eval_model(model, data_loader, criterion, device):
    # print("Model is eval on:", next(model.parameters()).device)
    model = model.eval()
    losses = []
    correct = 0
    correct_top5 = 0
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)

            correct += accuracy(outputs, labels)
            correct_top5 += top5_accuracy(outputs, labels)
            
            losses.append(loss.item())
    
    # print("Input_ids device:", input_ids.device)
    # print("Labels device:", labels.device)
    # print("outputs device:", outputs.device)
    
    return (correct / len(data_loader.dataset), 
            correct_top5 / len(data_loader.dataset),
              np.mean(losses))

Hardware

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

Evaluation utils

In [ ]:
class HSDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.descriptions = dataframe['GOODS_DESCRIPTION'].tolist()
        self.labels = dataframe['HS04'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.descriptions[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': self.labels[idx],
            'description': self.descriptions[idx]
        }
        return item


def predict_and_evaluate(
    model, tokenizer, unseen_sample, id2label,
    max_length=128, device='cpu', batch_size=32
):
    """
    Predicts the top 5 classes and their probabilities for an unseen sample using a given model and tokenizer.
    Calculates the accuracy for top 1 to top 5 predictions.
    Uses a DataLoader for GPU memory efficiency.
    """
    # Dataset & DataLoader
    dataset = HSDataset(unseen_sample, tokenizer, max_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.to(device)
    model.eval()

    all_top5_predicted_labels = []
    all_top5_predicted_probs = []
    all_true_labels = []
    all_descriptions = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probabilities = F.softmax(outputs, dim=1)
            top5_probs, top5_preds = torch.topk(probabilities, 5, dim=1)

            # Convert predictions and probabilities to lists
            for i in range(top5_preds.size(0)):
                pred_labels = [id2label[idx.item()] for idx in top5_preds[i]]
                pred_probs = [prob.item() for prob in top5_probs[i]]
                all_top5_predicted_labels.append(pred_labels)
                all_top5_predicted_probs.append(pred_probs)
            
            all_true_labels.extend(batch['label'])
            all_descriptions.extend(batch['description'])

    # Accuracy calculations
    accuracy_top1 = sum([
        all_true_labels[i] == all_top5_predicted_labels[i][0]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top2 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:2]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top3 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:3]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top4 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:4]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)
    accuracy_top5 = sum([
        all_true_labels[i] in all_top5_predicted_labels[i][:5]
        for i in range(len(all_true_labels))
    ]) / len(all_true_labels)

    print(f"Top-1 Accuracy: {accuracy_top1:.4f} %")
    print(f"Top-2 Accuracy: {accuracy_top2:.4f} %")
    print(f"Top-3 Accuracy: {accuracy_top3:.4f} %")
    print(f"Top-4 Accuracy: {accuracy_top4:.4f} %")
    print(f"Top-5 Accuracy: {accuracy_top5:.4f} %")

    metrics = {
    "top_1_acc": float(accuracy_top1),
    "top_2_acc": float(accuracy_top2),
    "top_3_acc": float(accuracy_top3),
    "top_4_acc": float(accuracy_top4),
    "top_5_acc": float(accuracy_top5),
    }

    # Results DataFrame
    results = pd.DataFrame({
        'Description': all_descriptions,
        'True Label': all_true_labels,
        'Top1': [labels[0] for labels in all_top5_predicted_labels],
        'Proba Top1': [probs[0] for probs in all_top5_predicted_probs],
        'Top2': [labels[1] for labels in all_top5_predicted_labels],
        'Proba Top2': [probs[1] for probs in all_top5_predicted_probs],
        'Top3': [labels[2] for labels in all_top5_predicted_labels],
        'Proba Top3': [probs[2] for probs in all_top5_predicted_probs],
        'Top4': [labels[3] for labels in all_top5_predicted_labels],
        'Proba Top4': [probs[3] for probs in all_top5_predicted_probs],
        'Top5': [labels[4] for labels in all_top5_predicted_labels],
        'Proba Top5': [probs[4] for probs in all_top5_predicted_probs],
    })
    results.set_index(unseen_sample.index, inplace=True)

    return results, metrics


Iterarion definitions

In [ ]:
import random
import joblib

fraction = 0.05
iterations = 10

min_val = 0
max_val = 999999999
random_seed = random.randint(min_val, max_val)

seeds = []

for iter in range(iterations):
    seed = random.randint(min_val, max_val)
    seeds.append(seed)

print("Random seeds for each iteration:")
print(seeds)  

out_dir = "results/distilbert/"
os.makedirs(out_dir, exist_ok=True)

Config columns

In [ ]:
target_col = 'HS04'

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

Config dataset

In [ ]:
max_length = 300
loader_batch_size = 32
shuffle = True

label_dir = "models/labels/"
os.makedirs(label_dir, exist_ok=True)

Config train

In [ ]:
import time
lr=2e-5

Iteration function

In [ ]:
def iterative_training(
    train_type: str,
    text_col: str,
    target_col: str,
    iterations: int,
    num_epochs: int,
    max_length: int,
    loader_batch_size: int,
    shuffle: bool,
    lr: float,
    fraction: float,
    out_dir: str,
    *,
    df: pd.DataFrame,
    seeds: Sequence[int],
    tokenizer,
    label_dir: str,
    fine_tune: bool = False,
    val_shuffle: Optional[bool] = None,
    num_workers: int = 0,
    device: Optional[torch.device] = None,
) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Entrena iterations modelos (distintos seeds) y devuelve:
      - scored_dfs: dict {model_name: results_df}
      - metrics_df: dataframe con métricas por modelo
    """

    if val_shuffle is None:
        val_shuffle = False  # en general no querés shuffle en validación

    if iterations > len(seeds):
        raise ValueError(f"iterations ({iterations}) > len(seeds) ({len(seeds)}).")

    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(label_dir, exist_ok=True)

    all_metrics = []
    scored_dfs = {}

    for iter_i in range(iterations):
        seed = seeds[iter_i]
        print(f"\n=== Iteration {iter_i+1}/{iterations} seed {seed} ===")

        model_name = f"DBERT_{train_type}_{text_col}_{target_col}_seed{seed}"
        print(f"Model name: {model_name}")

        train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

        # Label mappings (determinístico)
        unique_labels = sorted(df[target_col].astype(str).unique().tolist())
        label2id = {label: idx for idx, label in enumerate(unique_labels)}
        id2label = {idx: label for label, idx in label2id.items()}

        # Tokenizing training data
        train_encodings = tokenizer(
            list(train_df[text_col]),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        train_labels = torch.tensor([label2id[str(lbl)] for lbl in train_df[target_col].astype(str)])

        train_dataset = TokenizedDataset(train_encodings, train_labels)
        train_loader = DataLoader(
            train_dataset,
            batch_size=loader_batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
        )

        # Tokenizing val data
        val_encodings = tokenizer(
            list(val_df[text_col]),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        val_labels = torch.tensor([label2id[str(lbl)] for lbl in val_df[target_col].astype(str)])

        val_dataset = TokenizedDataset(val_encodings, val_labels)
        val_loader = DataLoader(
            val_dataset,
            batch_size=loader_batch_size,
            shuffle=val_shuffle,
            num_workers=num_workers,
        )

        # Save labels dictionary
        labels_dict = {"label2id": label2id, "id2label": id2label}
        labels_path = os.path.join(label_dir, f"labels_dict_{model_name}.json")
        with open(labels_path, "w") as f:
            json.dump(labels_dict, f, indent=4, ensure_ascii=False)

        # Model
        model = HSClassifier(n_classes=len(label2id), fine_tune=fine_tune)

        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Training on {device}")
        model = model.to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        history = {
            "train_loss": [],
            "train_acc": [],
            "train_top5_acc": [],
            "val_loss": [],
            "val_acc": [],
            "val_top5_acc": [],
        }

        for epoch in range(num_epochs):
            print(f"Epoch {epoch + 1}/{num_epochs}\n" + "-" * 10)
            start_time = time.time()

            train_acc, train_top5_acc, train_loss = train_epoch(
                model, train_loader, criterion, optimizer, device
            )
            print(f"Train loss {train_loss} accuracy {train_acc} top5_accuracy {train_top5_acc}")

            val_acc, val_top5_acc, val_loss = eval_model(
                model, val_loader, criterion, device
            )
            print(f"Validation loss {val_loss} accuracy {val_acc} top5_accuracy {val_top5_acc}")

            epoch_time = time.time() - start_time
            print(f"Epoch {epoch + 1} completed in {epoch_time/60:.2f} minutes.\n")

            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["train_top5_acc"].append(train_top5_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["val_top5_acc"].append(val_top5_acc)

        # Evaluate (sobre val_df)
        results, metrics = predict_and_evaluate(
            model,
            tokenizer,
            val_df,
            id2label,
            max_length=max_length,
            device=device,
        )

        scored_dfs[model_name] = results
        all_metrics.append({"model": model_name, **metrics})

        del model
        torch.cuda.empty_cache()

    metrics_df = pd.DataFrame(all_metrics).set_index("model").sort_index()

    # Guardar métricas (nombre correcto)
    metrics_path = os.path.join(out_dir, f"metrics_{train_type}_{text_col}_{target_col}.csv")
    metrics_df.to_csv(metrics_path, index=True)
    print(f"Saved metrics to {metrics_path}")

    return scored_dfs, metrics_df


#### Transfer learning

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

A- Raw descriptions

In [ ]:
train_type = "tf" # transfer learning - fixed encoder
fine_tune = False
num_epochs = 10

text_col = raw_col

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
)

B- Preproced descriptions

In [ ]:
text_col = prepro_col

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
)

C- Preproced + N-gram descriptions

In [ ]:
text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
)

### Fine-tuned model

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

A- Raw descriptions

In [ ]:
train_type = "ft" # fine tuned
fine_tune = True
num_epochs = 3

text_col = raw_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
)

B- Preproced descriptions

In [ ]:
train_type = "ft" # fine tuned
num_epochs = 3

text_col = prepro_col
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
)

C- Preproced + N-gram descriptions

In [ ]:
text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]
model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
)

### Partial fine-tuned


Fine-tuning last 2 layers

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

A- Raw descriptions

In [ ]:
train_type = "pft" # partial fine tuned
fine_tune = True
layers_to_finetune = 2
num_epochs = 5

text_col = raw_col

model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    n_finetune_layers=layers_to_finetune
)

B- Preproced descriptions

In [ ]:
text_col = prepro_col

model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    n_finetune_layers=layers_to_finetune
)

C- Preproced + N-gram descriptions

In [ ]:
text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]

model_family = f"DBERT_{train_type}_{text_col}_{target_col}"
print(f"=== {model_family} ===")

scored_dfs, metrics_df = iterative_training(
    train_type=train_type,
    text_col=text_col,
    target_col=target_col,
    iterations=iterations,
    num_epochs=num_epochs,
    max_length=max_length,
    loader_batch_size=loader_batch_size,
    shuffle=shuffle,
    lr=lr,
    fraction=fraction,
    out_dir=out_dir,
    df=df,
    seeds=seeds,
    tokenizer=tokenizer,
    label_dir=label_dir,
    fine_tune=fine_tune,
    n_finetune_layers=layers_to_finetune
)